In [16]:
import numpy as np

def dcg_at_k(scores, k=5):
    """
    计算DCG@k。
    参数:
    - scores: 一个包含相关性得分的列表或者numpy数组。
    - k: 考虑的顶部元素的数量。
    返回:
    - DCG@k的值。
    """
    scores = np.asfarray(scores)[:k]
    return np.sum((2**scores - 1) / np.log2(np.arange(2, scores.size + 2)))

def idcg_at_k(scores, k=5):
    """
    计算IDCG@k。
    参数:
    - scores: 一个包含相关性得分的列表或者numpy数组。
    - k: 考虑的顶部元素的数量。
    返回:
    - IDCG@k的值。
    """
    scores_sorted = np.sort(scores)[::-1]
    return dcg_at_k(scores_sorted, k)

def ndcg_at_k(scores, k=5):
    """
    计算NDCG@k。
    参数:
    - scores: 一个包含相关性得分的列表或者numpy数组。
    - k: 考虑的顶部元素的数量。
    返回:
    - NDCG@k的值。
    """
    dcg = dcg_at_k(scores, k)
    idcg = idcg_at_k(scores, k)
    return dcg / idcg if idcg > 0 else 0

# 示例使用
scores = [3, 2, 3, 0, 1]
k = 3
ndcg_score = ndcg_at_k(scores, k)
print(f"NDCG@{k}: {ndcg_score}")


NDCG@3: 0.9594535145926796


In [18]:
def hit_rate(recommended_items, actual_interactions):
    """
    计算命中率。
    参数:
    - recommended_items: 推荐系统给出的推荐项目列表。
    - actual_interactions: 用户实际交互（感兴趣）的项目列表。
    返回:
    - 命中率值。
    """
    hits = 0
    for item in recommended_items:
        if item in actual_interactions:
            hits += 1
    return hits / len(recommended_items) if recommended_items else 0

# 示例使用
recommended_items = ['item1', 'item2', 'item3', 'item4', 'item5']
actual_interactions = ['item2', 'item4', 'item6']

hit_rate_value = hit_rate(recommended_items, actual_interactions)
print(f"Hit Rate: {hit_rate_value}")


Hit Rate: 0.4


In [21]:
def mean_reciprocal_rank(rankings):
    """
    计算平均排名倒数（MRR）。
    
    参数:
    - rankings: 一个列表的列表，每个内部列表包含单个查询的相关项的排名，假设排名从1开始。
    
    返回:
    - MRR值。
    """
    rr_sum = 0.0
    for rank in rankings:
        if rank:
            rr_sum += 1.0 / min(rank)
    return rr_sum / len(rankings) if rankings else 0

# 测试MRR计算
rankings = [
    [1, 2, 3],  # 第一个查询，相关项在第一位
    [3, 4],  # 第二个查询，相关项在第三位
    [2, 3, 4],  # 第三个查询，相关项在第二位
]

mrr = mean_reciprocal_rank(rankings)
print(f"Mean Reciprocal Rank (MRR): {mrr:.3f}")


Mean Reciprocal Rank (MRR): 0.611


In [31]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import jieba

def bleu_score(true_answer : str, generate_answer : str, weights: tuple) -> float:
    # true_anser : 标准答案，str 类型
    # generate_answer : 模型生成答案，str 类型
    true_answers = list(jieba.cut(true_answer))
    generate_answers = list(jieba.cut(generate_answer))
    # 使用了平滑函数SmoothingFunction().method4来处理可能的0值情况，
    # 这对于避免由于缺乏高阶n-gram匹配导致的得分为0非常有用。
    smoothie = SmoothingFunction().method4
    bleu_score = sentence_bleu(true_answers, 
                               generate_answers, 
                               weights=weights, 
                               smoothing_function=smoothie)
    return bleu_score
    
    
# 标准答案
true_answer = "there is a cat on the mat"
# 模型生成答案
generate_answer = "the cat is on the mat"

# 计算BLEU得分
'''权重设置
1-gram BLEU得分：设置weights为(1, 0, 0, 0)
2-gram BLEU得分：设置weights为(0.5, 0.5, 0, 0)，表示1-gram和2-gram各占一半的权重。
3-gram BLEU得分：设置weights为(1/3, 1/3, 1/3, 0)
4-gram BLEU得分：设置weights为(0.25, 0.25, 0.25, 0.25)，这是默认设置，表示1-gram到4-gram均等考虑。
'''

weights = (0.5, 0.5, 0, 0)
score = bleu_score(true_answer, generate_answer, weights)
print(f"BLEU score: {score}")

BLEU score: 0.046689450558483385


In [1]:
# python 实现

# 参考摘要和生成摘要
reference = "The quick brown fox jumps over the lazy dog"
generated = "The brown fox jumped over the dog"

# 将摘要分割成单词
reference_tokens = reference.split()
generated_tokens = generated.split()

# 使用集合操作找到最长公共子序列的长度
lcs_set = set(reference_tokens) & set(generated_tokens)
lcs_length = len(lcs_set)

# 计算recall和precision
recall = lcs_length / len(reference_tokens)
precision = lcs_length / len(generated_tokens)

print(recall, precision)

0.6666666666666666 0.8571428571428571


In [5]:
from rouge import Rouge 

reference = "The quick brown fox jumps over the lazy dog"
generated = "The brown fox jumped over the dog"

rouge = Rouge()
scores = rouge.get_scores(generated, reference)
print(scores)

[{'rouge-1': {'r': 0.6666666666666666, 'p': 0.8571428571428571, 'f': 0.7499999950781251}, 'rouge-2': {'r': 0.25, 'p': 0.3333333333333333, 'f': 0.2857142808163266}, 'rouge-l': {'r': 0.6666666666666666, 'p': 0.8571428571428571, 'f': 0.7499999950781251}}]


In [4]:
import rouge

evaluate = rouge.Rouge(metrics=['rouge-n', 'rouge-l', 'rouge-w'], # n-gram重叠（ROUGE-N）、最长公共子序列（ROUGE-L）和基于权重的评分（ROUGE-W）
                       #max_n=4, # 在计算ROUGE-N时使用的最大n-gram长度是4-gram
                       limit_length=True, # 表示将对评估的文本长度进行限制
                       length_limit=100, # 如果limit_length为True，则这是文本长度的限制，意味着每个摘要将被截断到最多100个词。
                       length_limit_type='words', # 长度限制应用于词而不是字节
                       apply_avg=True, # 计算得分时，将对多个参考摘要进行平均
                       apply_best=False, # 如果设置为True，则只使用与hypothesis最匹配的参考摘要计算得分。此处设置为False，表示不使用这种方法。
                       alpha=0.5, # Default F1_score，用于计算F1得分时的权衡参数（alpha），用于平衡召回率（Recall）和精确度（Precision）的重要性。
                       weight_factor=1.2, # 特别用于ROUGE-W的参数，它调整了序列的权重，增加对更长匹配序列的评分。
                       stemming=True) # 如果为True，则在评估之前对词进行词干提取，这有助于将不同形式的单词（如复数形式）归一化为基本形式。

hypothesis = "the quick brown fox jumps over the lazy dog"
references = ["the fast brown fox jumps over the lazy dog"]
# 评估机器生成摘要的质量
scores = evaluate.get_scores(hypothesis, references)
print(scores)

TypeError: Rouge.__init__() got an unexpected keyword argument 'limit_length'